# Full Hebrew 4-Encoder Run (Aug + CRF + OOF)

This notebook runs the full thesis cross-comparison on the 4 Hebrew encoders only:
- dictabert
- berel
- hero
- alephbertgimmel

It performs:
1. Journal-mode lambda calibration (Exp04 and Exp10, resume-aware)
2. Full experiment sweep (non-CRF + CRF tracks, exp07 + exp07+aug)
3. Split consolidated error analysis workbooks
4. Unified protocol workbook with fold-paired tests

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/menachemsperka/thesis.git"
WORK_ROOT = Path("/content/thesis")
DRIVE_ROOT = Path("/content/drive/MyDrive/thesis_project")
OUTPUT_DIR = DRIVE_ROOT / "full_hebrew4_run"
MODELS = "dictabert,berel,hero,alephbertgimmel"
NUM_SEEDS = 20

EXPERIMENTS = (
    "01,04,05_ready,06_ready,"
    "06_svm_oof,06_svm_kernel_oof,06_nb_oof,06_lr_oof,06_rf_oof,06_mlp_oof,"
    "10_regular,10_cascade,10_fusion_ready,"
    "10_svm_oof,10_svm_kernel_oof,10_nb_oof,10_lr_oof,10_rf_oof,10_mlp_oof"
)

print("Configured output:", OUTPUT_DIR)
print("Models:", MODELS)
print("Num seeds:", NUM_SEEDS)

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

if not WORK_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORK_ROOT)], check=True)

os.chdir(WORK_ROOT)
print("Working directory:", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Colab-safe defaults and deterministic IO behavior
os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ["THESIS_CSV_ENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

# Optional: keep cache and outputs under Drive
os.environ["THESIS_CROSS_OUTPUT_DIR"] = str(OUTPUT_DIR)
print("THESIS_CROSS_OUTPUT_DIR=", os.environ["THESIS_CROSS_OUTPUT_DIR"])

In [ ]:
# Ensure training_seeds.json has at least NUM_SEEDS entries without invalidating prior completed runs
seed_path = OUTPUT_DIR / "training_seeds.json"
rng = random.Random(1729)

if seed_path.exists():
    data = json.loads(seed_path.read_text(encoding="utf-8"))
    seeds = data.get("seeds") if isinstance(data, dict) else data
    if not isinstance(seeds, list) or not seeds:
        raise RuntimeError(f"Invalid training seeds file: {seed_path}")
    seeds = [int(s) for s in seeds]
    seen = set(seeds)
    while len(seeds) < NUM_SEEDS:
        candidate = rng.randint(1, 2_147_483_647)
        if candidate not in seen:
            seeds.append(candidate)
            seen.add(candidate)
    if len(seeds) > NUM_SEEDS:
        seeds = seeds[:NUM_SEEDS]
else:
    seeds = []
    seen = set()
    while len(seeds) < NUM_SEEDS:
        candidate = rng.randint(1, 2_147_483_647)
        if candidate not in seen:
            seeds.append(candidate)
            seen.add(candidate)

seed_payload = {"num_seeds": len(seeds), "seeds": seeds}
seed_path.write_text(json.dumps(seed_payload, indent=2), encoding="utf-8")
print("Seed file:", seed_path)
print("Seed count:", len(seeds))

In [ ]:
# Quick free-space check for Drive output target
usage = shutil.disk_usage(str(DRIVE_ROOT))
free_gb = usage.free / (1024 ** 3)
total_gb = usage.total / (1024 ** 3)
print(f"Drive space: {free_gb:.1f} GB free / {total_gb:.1f} GB total")
if free_gb < 25:
    print("WARNING: Less than 25 GB free; full OOF caches may run out of space.")

In [ ]:
run_cmd = [
    sys.executable, "run_cross_data_model_comparison.py",
    "--journal-paper",
    "--resume",
    "--isolate-splits",
    "--output-dir", str(OUTPUT_DIR),
    "--exp07-source", "auto",
    "--models", MODELS,
    "--experiments", EXPERIMENTS,
    "--condition-sources", "exp07,exp07+aug",
    "--num-seeds", str(NUM_SEEDS),
    "--consolidated-error-analysis", "split",
    "--base-mode", "auto",
]

log_path = OUTPUT_DIR / "full_hebrew4_run.log"
print("Running command:")
print(" \
,
Log file:", log_path)

with log_path.open("w", encoding="utf-8") as logf:
    proc = subprocess.run(run_cmd, stdout=logf, stderr=subprocess.STDOUT, text=True)

print("Exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError(f"Run failed; inspect log: {log_path}")

In [ ]:
# Build unified OOF workbook with paired fold tests
unified_cmd = [
    sys.executable, "build_unified_protocol_table.py",
    "--oof-dir", str(OUTPUT_DIR),
    "--splits-dir", str(OUTPUT_DIR / "exp07" / "splits"),
    "--output", str(OUTPUT_DIR / "unified_protocol_table.xlsx"),
]
print("Running:", " \
,
,
Unified workbook ready:", OUTPUT_DIR / "unified_protocol_table.xlsx")

In [ ]:
from glob import glob

def newest(pattern: str):
    matches = sorted(glob(pattern))
    return matches[-1] if matches else None

cross_latest = OUTPUT_DIR / "cross_comparison_latest.xlsx"
cross_json = OUTPUT_DIR / "cross_comparison_latest.json"
exp04_grid = newest(str(OUTPUT_DIR / "exp04_lambda_grid_cache" / "loss_weight_grid_*.xlsx"))
exp10_grid = newest(str(OUTPUT_DIR / "exp10_lambda_grid_cache" / "loss_weight_grid_*.xlsx"))
exp04_sel = OUTPUT_DIR / "exp04_lambda_grid_cache" / "exp04_lambda_selection.json"
exp10_sel = OUTPUT_DIR / "exp10_lambda_grid_cache" / "exp10_lambda_selection.json"
err_non = newest(str(OUTPUT_DIR / "consolidated_error_analysis_non_exp10_*.xlsx"))
err_crf = newest(str(OUTPUT_DIR / "consolidated_error_analysis_exp10_*.xlsx"))
unified_xlsx = OUTPUT_DIR / "unified_protocol_table.xlsx"
log_path = OUTPUT_DIR / "full_hebrew4_run.log"

print("Main results workbook:", cross_latest)
print("Main results json:", cross_json)
print("Exp04 grid workbook:", exp04_grid)
print("Exp10 grid workbook:", exp10_grid)
print("Exp04 selection json:", exp04_sel)
print("Exp10 selection json:", exp10_sel)
print("Error analysis non-CRF:", err_non)
print("Error analysis CRF:", err_crf)
print("Unified protocol workbook:", unified_xlsx)
print("Run log:", log_path)

required = [cross_latest, cross_json, exp04_sel, exp10_sel, unified_xlsx]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise RuntimeError("Missing required outputs: " + ", ".join(missing))
if err_non is None or err_crf is None:
    raise RuntimeError("Missing consolidated error-analysis workbook(s).")
print("All required outputs found.")